# DMV Practical 3 – Telco Customer Data Preprocessing

**Aim:** Inspect a customer churn dataset, handle missing values and outliers, standardize categorical values, create tenure and service features, and examine churn distribution.

Uses the Telco customer churn CSV referenced in the PDF. If the source URL is unavailable, upload the same CSV file when prompted.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://raw.githubusercontent.com/Giskard-AI/examples/main/datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv"
try:
    df = pd.read_csv(url)
except Exception:
    print("Could not load dataset from URL. Upload WA_Fn-UseC_-Telco-Customer-Churn.csv.")
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))

print("Dataset shape:", df.shape)
display(df.head())
df.info()
display(df.describe(include="all").T)

In [ ]:
# Missing values and duplicates
print("Missing values:\n", df.isnull().sum())
print("Missing percentage:\n", (df.isnull().sum()/len(df)*100).round(2))
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()

# Inspect categorical values
categorical_columns = df.select_dtypes(include="object").columns
for col in categorical_columns:
    print("\n", col)
    print(df[col].value_counts().head(20))

In [ ]:
# Standardize text and convert numeric columns
for col in categorical_columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"": np.nan, "nan": np.nan})

if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
if "tenure" in df.columns:
    df["tenure"] = pd.to_numeric(df["tenure"], errors="coerce")

numeric_columns = df.select_dtypes(include=np.number).columns
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    if df[col].isnull().sum():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining missing values:", df.isnull().sum().sum())

In [ ]:
# Outlier handling with IQR clipping
numeric_columns = df.select_dtypes(include=np.number).columns
for col in numeric_columns:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    df[col] = df[col].clip(q1 - 1.5*iqr, q3 + 1.5*iqr)
print("Outliers handled using IQR capping.")

In [ ]:
# Tenure groups and service count
if "tenure" in df.columns:
    df["TenureGroup"] = pd.cut(
        df["tenure"], bins=[-1, 12, 24, 48, 72, np.inf],
        labels=["0-1 Year", "1-2 Years", "2-4 Years", "4-6 Years", "6+ Years"]
    )

service_columns = [
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies"
]
existing_services = [c for c in service_columns if c in df.columns]
df["ServiceCount"] = 0
for col in existing_services:
    df["ServiceCount"] += df[col].astype(str).str.lower().isin(["yes", "fiber optic", "dsl"]).astype(int)

display(df.head())

In [ ]:
# Churn distribution
if "Churn" in df.columns:
    print(df["Churn"].value_counts())
    print((df["Churn"].value_counts(normalize=True)*100).round(2))
    df["Churn"].value_counts().plot(kind="bar", figsize=(5,3), title="Customer Churn Distribution")
    plt.xlabel("Churn"); plt.ylabel("Number of Customers")
    plt.tight_layout(); plt.show()

# Numerical distributions
for col in df.select_dtypes(include=np.number).columns:
    plt.figure(figsize=(5,3))
    sns.boxplot(x=df[col])
    plt.title("Box Plot - " + col)
    plt.tight_layout()
    plt.show()

In [ ]:
# ========== DMV PRACTICAL 3: TELCO CUSTOMER DATA PREPROCESSING ==========

# AIM: Clean customer data, handle missing values/outliers,
# create new features and analyze customer churn.
# FLOW: Load -> Inspect -> Clean -> Handle Outliers ->
#       Analyze Churn -> Create Features

# THEORY:
# Data Preprocessing = preparing raw data for analysis.
# Churn = customer stops using a company's service.
# Missing Value = empty/unavailable data.
# Outlier = unusually high or low value.
# Feature Engineering = creating useful new columns.
# IQR = Interquartile Range (Q3 - Q1).
# Median = middle value; Mode = most frequent value.

# LIBRARIES:
# pandas(pd) = data handling; numpy(np) = numerical operations.
# matplotlib(plt) = graphs; seaborn(sns) = statistical plots.

# DATA LOADING:
# try = attempts to load CSV from URL.
# except Exception = handles loading errors.
# pd.read_csv(url) = loads CSV into DataFrame.
# files.upload() = allows manual upload in Google Colab.
# next(iter(uploaded)) = gets first uploaded filename.
# df.shape = (rows, columns); head() = first 5 rows.
# info() = column types and non-null counts.
# describe(include="all") = statistics for all columns.
# .T = transposes table (swaps rows and columns).
# display() = displays formatted table.

# MISSING VALUES & DUPLICATES:
# isnull().sum() = missing values per column.
# len(df) = total rows.
# isnull().sum()/len(df)*100 = missing percentage.
# round(2) = rounds to 2 decimal places.
# duplicated().sum() = number of duplicate rows.
# drop_duplicates() = removes duplicate rows.
# copy() = creates independent DataFrame copy.

# CATEGORICAL DATA:
# select_dtypes(include="object") = selects text columns.
# .columns = gets column names.
# value_counts() = frequency of each category.
# head(20) = first 20 values.
# astype(str) = converts values into strings.
# str.strip() = removes extra spaces.
# replace({"":np.nan,"nan":np.nan}) = converts blanks to NaN.
# np.nan = missing value.

# NUMERICAL DATA:
# pd.to_numeric(...,errors="coerce") = converts text to numbers.
# errors="coerce" = invalid values become NaN.
# select_dtypes(include=np.number) = selects numerical columns.
# fillna(median()) = fills missing numbers with median.
# fillna(mode()[0]) = fills missing categories with most common value.
# mode()[0] = first most frequent value.
# isnull().sum().sum() = total remaining missing values.

# OUTLIER HANDLING (IQR METHOD):
# quantile([0.25,0.75]) = calculates Q1 and Q3.
# Q1 = 25th percentile; Q3 = 75th percentile.
# IQR = Q3 - Q1.
# Lower Limit = Q1 - 1.5*IQR.
# Upper Limit = Q3 + 1.5*IQR.
# clip(lower,upper) = caps extreme values within limits.
# NOTE: Capping changes outliers; it does NOT delete rows.

# CHURN ANALYSIS:
# if "Churn" in df.columns = checks whether column exists.
# value_counts() = number of Yes/No churn customers.
# value_counts(normalize=True) = proportions instead of counts.
# *100 = converts proportions into percentages.
# plot(kind="bar",figsize=(5,3),title="...") = bar chart.
# xlabel() = X-axis name; ylabel() = Y-axis name.
# tight_layout() = adjusts spacing; show() = displays plot.

# NUMERICAL VISUALIZATION:
# for col in ... = loops through numerical columns.
# plt.figure(figsize=(5,3)) = sets graph size in inches.
# sns.boxplot(x=df[col]) = displays numerical distribution.
# Boxplot shows median, quartiles and possible outliers.

# FEATURE ENGINEERING:
# pd.cut() = divides numerical values into categories.
# pd.cut(df["tenure"],bins=[-1,12,24,48,72,np.inf],labels=[...])
# bins = group boundaries; labels = names of tenure groups.
# np.inf = infinity (no upper limit).
# TenureGroup = customer duration categories.
# service_columns = list of service-related column names.
# [c for c in service_columns if c in df.columns]
# = selects only service columns present in dataset.
# ServiceCount = number of selected active services.
# str.lower() = converts text to lowercase.
# isin(["yes","fiber optic","dsl"]) = checks active-service values.
# astype(int) = converts True/False into 1/0.
# += = adds active service indicator to ServiceCount.
# NOTE: "No internet service" and "No phone service" count as 0.

# ========== 4 IMPORTANT VIVA QUESTIONS ==========

# Q1. What is Data Preprocessing?
# Ans: Cleaning and transforming raw data to make it usable.

# Q2. What is Customer Churn?
# Ans: When a customer stops using a company's services.

# Q3. How are missing values and outliers handled?
# Ans: Median fills missing numbers, mode fills categories,
#      and IQR clipping caps extreme numerical values.

# Q4. What is Feature Engineering?
# Ans: Creating useful new features from existing data.
# Example: TenureGroup and ServiceCount.

# FINAL VIVA EXPLANATION:
# "We load the Telco customer churn dataset, inspect missing
# values and duplicates, clean text and numerical columns,
# fill missing values using median/mode, handle outliers using
# IQR clipping, analyze churn using graphs, and create
# TenureGroup and ServiceCount features for further analysis."